In [ ]:
import os
from fastrtc import (ReplyOnPause, Stream, get_stt_model, get_tts_model)
import google.generativeai as genai
from dotenv import load_dotenv
load_dotenv()

# Gemini 클라이언트 초기화
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

stt_model = get_stt_model()
tts_model = get_tts_model()
model = genai.GenerativeModel("gemini-1.5-flash")  # 또는 "gemini-1.5-pro"

def echo(audio):
    prompt = stt_model.stt(audio)
    response = model.generate_content(prompt)
    reply = response.text

    for audio_chunk in tts_model.stream_tts_sync(reply):
        yield audio_chunk

stream = Stream(ReplyOnPause(echo), modality="audio", mode="send-receive")
stream.ui.launch()

In [ ]:
# 채팅 구현해보기

from flask import Flask, render_template
from flask_socketio import SocketIO
import os
from datetime import datetime

app = Flask(__name__, template_folder=os.getcwd()+'/templates/')
socketio = SocketIO(app)

@app.route('/')
def index():
    return render_template('chat.html')

@socketio.on('message')
def handle_message(data):
    print('Received message:', data)
    # Add server timestamp to the message data
    data['timestamp'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    socketio.emit('message', data)

if __name__ == '__main__':
    socketio.run(app, allow_unsafe_werkzeug=True)
    # socketio.run(app, host='l', port=4000, debug=True)

In [ ]:
import io
import base64
import speech_recognition as sr
from flask import Flask, render_template
from flask_socketio import SocketIO, emit

app = Flask(__name__)
socketio = SocketIO(app)

@app.route("/")
def index():
    return render_template("real-time.html")

# 음성 데이터 수신
@socketio.on("audio_stream")
def handle_audio(data):
    audio_bytes = base64.b64decode(data["audio"])
    recognizer = sr.Recognizer()                                                                                                                                                                                                                                                       
    with sr.AudioFile(io.BytesIO(audio_bytes)) as source:
        audio_data = recognizer.record(source)
        try:
            text = recognizer.recognize_google(audio_data, language="ko-KR")
            emit("text_result", {"user": data["user"], "text": text}, broadcast=True)
        except sr.UnknownValueError:
            pass

if __name__ == "__main__":
    socketio.run(app, host="0.0.0.0", port=5002, debug=False, allow_unsafe_werkzeug=True)

Werkzeug appears to be used in a production deployment. Consider switching to a production web server instead.


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5002
 * Running on http://172.29.83.136:5002
Press CTRL+C to quit
127.0.0.1 - - [02/Nov/2025 10:51:39] "GET / HTTP/1.1" 200 -


In [ ]:
# 비동기 방식

import io
import base64
import speech_recognition as sr
from flask import Flask, render_template, request
from flask_socketio import SocketIO, emit
import eventlet
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = Flask(__name__)
# Use eventlet for async support
socketio = SocketIO(app, async_mode='eventlet', cors_allowed_origins="*")

@app.route("/")
def index():
    logger.info("Serving real-time.html")
    return render_template("real-time.html")

# 음성 데이터 수신
@socketio.on("audio_stream")
def handle_audio(data):
    try:
        logger.info(f"Received audio data from user: {data.get('user', 'Unknown')}")
        audio_bytes = base64.b64decode(data["audio"])
        recognizer = sr.Recognizer()
        
        with sr.AudioFile(io.BytesIO(audio_bytes)) as source:
            audio_data = recognizer.record(source)
            try:
                text = recognizer.recognize_google(audio_data, language="ko-KR")
                logger.info(f"Recognized text: {text}")
                # 실시간으로 텍스트 결과 전송
                emit("text_result", {"user": data["user"], "text": text}, broadcast=True)
            except sr.UnknownValueError:
                # 음성 인식 실패 시 처리
                logger.warning("Could not understand audio")
                emit("text_result", {"user": data["user"], "text": "[음성 인식 실패]"}, broadcast=True)
            except sr.RequestError as e:
                # 인터넷 연결 문제 등 처리
                logger.error(f"Request error: {e}")
                emit("text_result", {"user": data["user"], "text": f"[요청 오류: {e}]"}, broadcast=True)
    except Exception as e:
        logger.error(f"Error processing audio: {e}")
        emit("text_result", {"user": data.get("user", "Unknown"), "text": f"[처리 오류: {e}]"}, broadcast=True)

# Handle connection events
@socketio.on('connect')
def handle_connect():
    logger.info(f"Client connected: {request.sid}")

@socketio.on('disconnect')
def handle_disconnect():
    logger.info(f"Client disconnected: {request.sid}")

if __name__ == "__main__":
    # For development only - use Gunicorn with eventlet workers in production
    logger.info("Starting async speech recognition app on port 5002")
    socketio.run(app, host="0.0.0.0", port=5002, debug=False)

In [ ]:
# fastrtc test # 1

import asyncio
import base64
import os
import time
from io import BytesIO

import gradio as gr
import numpy as np
from google import genai
from fastrtc import (
    AsyncAudioVideoStreamHandler,
    WebRTC,
    async_aggregate_bytes_to_16bit,
    VideoEmitType,
    AudioEmitType,
    get_twilio_turn_credentials,
)
from PIL import Image


def encode_audio(data: np.ndarray) -> dict:
    """Encode Audio data to send to the server"""
    return {"mime_type": "audio/pcm", "data": base64.b64encode(data.tobytes()).decode("UTF-8")}


def encode_image(data: np.ndarray) -> dict:
    with BytesIO() as output_bytes:
        pil_image = Image.fromarray(data)
        pil_image.save(output_bytes, "JPEG")
        bytes_data = output_bytes.getvalue()
    base64_str = str(base64.b64encode(bytes_data), "utf-8")
    return {"mime_type": "image/jpeg", "data": base64_str}


class GeminiHandler(AsyncAudioVideoStreamHandler):
    def __init__(
        self, expected_layout="mono", output_sample_rate=24000, output_frame_size=480
    ) -> None:
        super().__init__(
            expected_layout,
            output_sample_rate,
            output_frame_size,
            input_sample_rate=16000,
        )
        self.audio_queue = asyncio.Queue()
        self.video_queue = asyncio.Queue()
        self.quit = asyncio.Event()
        self.session = None
        self.last_frame_time = 0

    def copy(self) -> "GeminiHandler":
        return GeminiHandler(
            expected_layout=self.expected_layout,
            output_sample_rate=self.output_sample_rate,
            output_frame_size=self.output_frame_size,
        )
    
    async def video_receive(self, frame: np.ndarray):
        if self.session:
            # send image every 1 second
            if time.time() - self.last_frame_time > 1:
                self.last_frame_time = time.time()
                await self.session.send(encode_image(frame))
                if self.latest_args[2] is not None:
                    await self.session.send(encode_image(self.latest_args[2]))
        self.video_queue.put_nowait(frame)
    
    async def video_emit(self) -> VideoEmitType:
        return await self.video_queue.get()

    async def connect(self, api_key: str):
        if self.session is None:
            client = genai.Client(api_key=api_key, http_options={"api_version": "v1alpha"})
            config = {"response_modalities": ["AUDIO"]}
            async with client.aio.live.connect(
                model="gemini-2.0-flash-exp", config=config
            ) as session:
                self.session = session
                asyncio.create_task(self.receive_audio())
                await self.quit.wait()

    async def generator(self):
        while not self.quit.is_set():
            turn = self.session.receive()
            async for response in turn:
                if data := response.data:
                    yield data
    
    async def receive_audio(self):
        async for audio_response in async_aggregate_bytes_to_16bit(
            self.generator()
        ):
            self.audio_queue.put_nowait(audio_response)

    async def receive(self, frame: tuple[int, np.ndarray]) -> None:
        _, array = frame
        array = array.squeeze()
        audio_message = encode_audio(array)
        if self.session:
            await self.session.send(audio_message)

    async def emit(self) -> AudioEmitType:
        if not self.args_set.is_set():
            await self.wait_for_args()
        if self.session is None:
            asyncio.create_task(self.connect(self.latest_args[1]))
        array = await self.audio_queue.get()
        return (self.output_sample_rate, array)

    def shutdown(self) -> None:
        self.quit.set()
        self.connection = None
        self.args_set.clear()
        self.quit.clear()



css = """
#video-source {max-width: 600px !important; max-height: 600 !important;}
"""

with gr.Blocks(css=css) as demo:
    gr.HTML(
        """
    <div style='display: flex; align-items: center; justify-content: center; gap: 20px'>
        <div style="background-color: var(--block-background-fill); border-radius: 8px">
            <img src="https://www.gstatic.com/lamda/images/gemini_favicon_f069958c85030456e93de685481c559f160ea06b.png" style="width: 100px; height: 100px;">
        </div>
        <div>
            <h1>INFO: gradio_webrtc's new home is FastRTC. Use demo <a href="https://huggingface.co/spaces/fastrtc/gemini-audio-video" target="_blank"/>here</a></h1>
            <h1>Gen AI SDK Voice Chat</h1>
            <p>Speak with Gemini using real-time audio + video streaming</p>
            <p>Powered by <a href="https://gradio.app/">Gradio</a> and <a href=https://fastrtc.org/>FastRTC</a>⚡️</p>
            <p>Get an API Key <a href="https://support.google.com/googleapi/answer/6158862?hl=en">here</a></p>
        </div>
    </div>
    """
    )
    with gr.Row() as api_key_row:
        api_key = gr.Textbox(label="API Key", type="password", placeholder="Enter your API Key", value=os.getenv("GOOGLE_API_KEY"))
    with gr.Row(visible=False) as row:
        with gr.Column():
            webrtc = WebRTC(
                label="Video Chat",
                modality="audio-video",
                mode="send-receive",
                elem_id="video-source",
                rtc_configuration={
                    "iceServers": [
                        {"urls": ["stun:stun.l.google.com:19302"]},
                        {"urls": ["stun:stun1.l.google.com:19302"]}
                    ]
                },
                icon="https://www.gstatic.com/lamda/images/gemini_favicon_f069958c85030456e93de685481c559f160ea06b.png",
                pulse_color="rgb(35, 157, 225)",
                icon_button_color="rgb(35, 157, 225)",
            )
        with gr.Column():
            image_input = gr.Image(label="Image", type="numpy", sources=["upload", "clipboard"])

        webrtc.stream(
            GeminiHandler(),
            inputs=[webrtc, api_key, image_input],
            outputs=[webrtc],
            time_limit=90,
            concurrency_limit=2,
        )
        api_key.submit(
        lambda: (gr.update(visible=False), gr.update(visible=True)),
        None,
        [api_key_row, row],
    )


if __name__ == "__main__":
    demo.launch(share=True)

In [ ]:
from fastrtc import Stream
import gradio as gr
import numpy as np

def detection(image, slider):
    return np.flip(image, axis=0)

stream = Stream(
    handler=detection,
    modality="video",
    mode="send-receive",
    additional_inputs=[
        gr.Slider(minimum=0, maximum=1, step=0.01, value=0.3)
    ],
)

# HTTPS 설정
stream.ui.launch(
    server_name="0.0.0.0",   # 외부 접근 허용 (옵션)
    server_port=5008,
    ssl_certfile="localhost.pem",
    ssl_keyfile="localhost-key.pem"
)

if __name__ == "__main__":
    demo.launch(share=True)

In [16]:


import subprocess
import json
import os

video_ids = ["q_8u_vV3gQ8", "FUtuQvEB5ZE", "7zdOOs0EKsU"]
all_scripts = {}

for vid in video_ids:
    try:
        # yt-dlp로 자막 다운로드
        cmd = f'yt-dlp --write-auto-sub --sub-lang ko --skip-download --output "{vid}.%(ext)s" "https://www.youtube.com/watch?v={vid}"'
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        
        # vtt 파일 읽기
        vtt_file = f"{vid}.ko.vtt"
        if os.path.exists(vtt_file):
            with open(vtt_file, 'r', encoding='utf-8') as f:
                content = f.read()
            all_scripts[vid] = content
            os.remove(vtt_file)  # 임시 파일 삭제
        else:
            all_scripts[vid] = "❌ 자막 파일을 찾을 수 없음"
    except Exception as e:
        all_scripts[vid] = f"❌ 오류: {str(e)}"

# 텍스트 파일 저장
with open('youtube_scripts.txt', 'w', encoding='utf-8') as f:
    for vid, data in all_scripts.items():
        f.write(f'=== {vid} ===\n')
        f.write(f'{data}\n\n')

print('✅ youtube_scripts.txt 파일 저장 완료')
print(f'처리된 영상: {len(all_scripts)}개')

✅ youtube_scripts.txt 파일 저장 완료
처리된 영상: 3개
